# Atividade Avaliativa CDIA - Pipeline de Big Data

Pipeline em camadas seguindo a **Arquitetura Medallion** usando **Apache Spark**, **Silo** (S3 local) e **Delta Lake**.  

**Etapas:**
1. Inicialização do Spark com Silo (Data Lake local);
2. Validação da conexão;
3. Camada Bronze;  
  3.1. Validação e Leitura da Camada Bronze;  
  3.2. Histórico e Versionamento do Delta Lake (Time Travel);
4. Camada Silver;
5. Camada Gold.

## 1. Inicialização do Spark com Suporte ao Silo

Para conectar o Spark a um serviço compatível com S3 (Container do Silo rodando na porta 9000) é necessário:
1. **JARs do Hadoop AWS e AWS Java SDK**: Fornecem as classes Java necessárias para entender a URI `s3a://` (`hadoop-aws` e `aws-java-sdk-bundle`);
2. **Endpoint local**: Aponta para `http://localhost:9000` em vez de um servidor na AWS;
3. **Credenciais**: `admin` / `password123` via `SimpleAWSCredentialsProvider`;
4. **Path-style access**: Obrigatório para MinIO/Silo (`http://endpoint/bucket` em vez de `http://bucket.endpoint`);
5. **Desativação de SSL**: O container local é HTTP puro.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta.tables import DeltaTable

# Inicializa o SparkSession com suporte integrado a Silo e Delta Lake
spark = (
    SparkSession.builder
    .appName('Pipeline_Big_Data_CDIA')
    
    # 1. Download/Inclusão dos JARs: Hadoop AWS + AWS SDK + Delta Lake 3.2.0
    .config(
        'spark.jars.packages',
        'org.apache.hadoop:hadoop-aws:3.3.4,'
        'com.amazonaws:aws-java-sdk-bundle:1.12.262,'
        'io.delta:delta-spark_2.12:3.2.0'
    )
    
    # 2. Conexão com o Silo (porta 9000 local)
    .config('spark.hadoop.fs.s3a.endpoint', 'http://localhost:9000')
    
    # 3. Credenciais de autenticação
    .config('spark.hadoop.fs.s3a.access.key', 'admin')
    .config('spark.hadoop.fs.s3a.secret.key', 'password123')
    .config(
        'spark.hadoop.fs.s3a.aws.credentials.provider',
        'org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider'
    )
    
    # 4. Formato de acesso por caminho (Path Style) e desativação de SSL (HTTP local)
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')
    .config('spark.hadoop.fs.s3a.connection.ssl.enabled', 'false')
    
    # 5. Mapeamento da classe de sistema de arquivos do S3A
    .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
    
    # 6. Configurações do Delta Lake
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config(
        'spark.sql.catalog.spark_catalog',
        'org.apache.spark.sql.delta.catalog.DeltaCatalog'
    )
    
    .getOrCreate()
)

print('✓ SparkSession conectada ao Silo e Delta Lake com sucesso!')
print(f'✓ Aplicação: {spark.sparkContext.appName}')
print(f'✓ Spark Web UI: http://localhost:4040')
print(f'✓ Silo Console UI: http://localhost:9001')

:: loading settings :: url = jar:file:/home/raisson/cdia-trabalho-big-data/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/raisson/.ivy2/cache
The jars for the packages stored in: /home/raisson/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-a9b7c11e-8f05-47d2-8b27-f8c6b03811ea;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 278ms :: artifacts dl 12ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr

✓ SparkSession conectada ao Silo e Delta Lake com sucesso!
✓ Aplicação: Pipeline_Big_Data_CDIA
✓ Spark Web UI: http://localhost:4040
✓ Silo Console UI: http://localhost:9001


## 2. Teste de Conexão
#### Leitura da Camada Landing

Lemos os dados brutos gerados pelo `gerador-dados.ipynb` no caminho `s3a://lakehouse/landing/vendas_cruas_csv`. 
Na Landing, o dado ainda está em CSV com separador `;` e todos os tipos de dados são tratados como texto (`string`).

In [2]:
CAMINHO_LANDING = 's3a://lakehouse/landing/vendas_cruas_csv'

df_landing = (
    spark.read
    .format('csv')
    .option('header', 'true')
    .option('sep', ';')
    .load(CAMINHO_LANDING)
)

print(f'Total de registros na Landing Zone: {df_landing.count():,}')
df_landing.printSchema()
df_landing.show(5, truncate=False)

26/11/22 15:46:40 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


Total de registros na Landing Zone: 1,010,000
root
 |-- id_venda: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- filial_uf: string (nullable = true)
 |-- forma_pagamento: string (nullable = true)
 |-- quantidade: string (nullable = true)
 |-- preco_unitario: string (nullable = true)
 |-- desconto: string (nullable = true)

+--------+-----------+---------+---------------+----------+--------------+--------+
|id_venda|categoria  |filial_uf|forma_pagamento|quantidade|preco_unitario|desconto|
+--------+-----------+---------+---------------+----------+--------------+--------+
|875000  |NULL       |RS       |Boleto         |1         |189,76        |0.0     |
|875001  |livros     |SC       |PIX            |2         |-8.47         |0.05    |
|875002  |INFORMÁTICA|PR       |Cartão         |3         |363,41        |0.0     |
|875003  |Casa       |SP       |Boleto         |4         |245.49        |0.0     |
|875004  |Esporte    |RS       |PIX            |5         |62,7

## 3. Camada Bronze

Nenhuma limpeza, filtro ou transformação foi realizada para a camada bronze, o dado cru foi preservado da forma como chegou, apenas as colunas de metadados de auditoria `_data_ingestao` (timestamp do momento da ingestão do registro) e `_arquivo_origem` (nome do arquivo de origem) foram adicionadas, por fim foi realizada a gravação em formato Delta.

In [3]:
# Adição dos metadados de auditoria
df_bronze = (
    df_landing
    .withColumn('_data_ingestao', F.current_timestamp())
    .withColumn('_arquivo_origem', F.input_file_name())
)

# Gravação da tabela Delta na camada Bronze
CAMINHO_BRONZE = 's3a://lakehouse/bronze/vendas'

(
    df_bronze.write
    .format('delta')
    .mode('overwrite')
    .save(CAMINHO_BRONZE)
)

print('✓ Dados gravados com sucesso na Camada Bronze.')

26/11/22 15:46:54 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/11/22 15:46:59 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


✓ Dados gravados com sucesso na Camada Bronze.


### 3.1 Validação e Leitura da Camada Bronze

Lemos os dados gravados a partir do caminho Delta para conferir o schema e os metadados gerados.

In [4]:
df_bronze_read = spark.read.format('delta').load(CAMINHO_BRONZE)

print(f'Total de registros na Camada Bronze: {df_bronze_read.count():,}')
df_bronze_read.printSchema()
df_bronze_read.show(5, truncate=False)

Total de registros na Camada Bronze: 1,010,000
root
 |-- id_venda: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- filial_uf: string (nullable = true)
 |-- forma_pagamento: string (nullable = true)
 |-- quantidade: string (nullable = true)
 |-- preco_unitario: string (nullable = true)
 |-- desconto: string (nullable = true)
 |-- _data_ingestao: timestamp (nullable = true)
 |-- _arquivo_origem: string (nullable = true)

+--------+-----------+---------+---------------+----------+--------------+--------+--------------------------+-------------------------------------------------------------------------------------------------+
|id_venda|categoria  |filial_uf|forma_pagamento|quantidade|preco_unitario|desconto|_data_ingestao            |_arquivo_origem                                                                                  |
+--------+-----------+---------+---------------+----------+--------------+--------+--------------------------+--------------------------

### 3.2 Histórico e Versionamento do Delta Lake (Time Travel)

O Delta Lake armazena um log de transações (`_delta_log`). É inspecionar o histórico com `DeltaTable.forPath()` para auditar todas as operações realizadas.  
Exemplo apenas para a camada bronze.

In [5]:
tabela_delta_bronze = DeltaTable.forPath(spark, CAMINHO_BRONZE)
tabela_delta_bronze.history().select('version', 'timestamp', 'operation', 'operationMetrics').show(truncate=False)

+-------+-------------------+---------+---------------------------------------------------------------------+
|version|timestamp          |operation|operationMetrics                                                     |
+-------+-------------------+---------+---------------------------------------------------------------------+
|1      |2026-11-22 15:47:03|WRITE    |{numFiles -> 8, numOutputRows -> 1010000, numOutputBytes -> 10057382}|
|0      |2026-11-22 15:23:59|WRITE    |{numFiles -> 8, numOutputRows -> 1010000, numOutputBytes -> 10057390}|
+-------+-------------------+---------+---------------------------------------------------------------------+

